## 自定义输出解析器

在某些情况下，您可能希望实现自定义解析器以将模型输出构造为自定义格式。

有两种方法可以实现自定义解析器：
- 在 LCEL 中使用 RunnableLambda 或 RunnableGenerator – 我们强烈建议大多数用例使用此方法
- 通过从基类之一继承进行解析——这是困难方法

这两种方法之间的差异大多是表面的，主要在于触发哪些回调（例如， on_chain_start 与 on_parser_start ）

### 可运行的 Lambda 和生成器

推荐的解析方法是使用可运行的 lambda 和可运行的生成器！

在这里，我们将进行一个简单的解析，反转模型输出的大小写。

例如，如果模型输出：“Meow”，解析器将生成“mEOW”。


In [2]:
from typing import Iterable
from langchain_core.messages import AIMessage, AIMessageChunk

from langchain_openai import ChatOpenAI, OpenAI
openai_api_key = "EMPTY"
openai_api_base = "http://127.0.0.1:1234/v1"
model = ChatOpenAI(
    openai_api_key=openai_api_key,
    openai_api_base=openai_api_base,
    temperature=0.3,
)


def parse(ai_message: AIMessage) -> str:
    """Parse the AI message."""
    return ai_message.content.swapcase()


chain = model | parse
chain.invoke("Hello")

'<THINK>\noKAY, THE USER SAID "hELLO" AND i NEED TO RESPOND. lET ME START BY ACKNOWLEDGING THEIR GREETING. i SHOULD MAKE SURE MY RESPONSE IS FRIENDLY AND ENGAGING. mAYBE ADD A BIT OF ENTHUSIASM TO KEEP THE CONVERSATION GOING. i SHOULD ALSO CHECK IF THERE\'S ANYTHING ELSE THEY MIGHT WANT TO KNOW. lET ME THINK ABOUT COMMON QUESTIONS PEOPLE MIGHT HAVE AFTER SAYING HELLO. tHEY MIGHT BE STARTING A CONVERSATION, LOOKING FOR HELP, OR JUST WANTING TO CHAT. i SHOULD KEEP THE RESPONSE OPEN-ENDED TO INVITE FURTHER INTERACTION. lET ME MAKE SURE MY LANGUAGE IS WARM AND APPROACHABLE. aLRIGHT, TIME TO PUT THAT INTO A NATURAL RESPONSE.\n</THINK>\n\nhELLO! hOW\'S EVERYTHING GOING? 😊 i\'M GLAD YOU\'RE HERE TO CHAT! wHETHER YOU WANT TO START A CONVERSATION, ASK A QUESTION, OR JUST SHARE SOMETHING, i\'M HERE TO HELP. lET ME KNOW HOW i CAN ASSIST YOU TODAY! 🌟'

In [3]:
for chunk in chain.stream("tell me a story"):
    print(chunk, end="")

<THINK>
oKAY, THE USER WANTS ME TO TELL A STORY. lET ME THINK ABOUT WHAT KIND OF STORY THEY MIGHT LIKE. tHEY COULD BE LOOKING FOR SOMETHING ENGAGING AND MAYBE A BIT ADVENTUROUS. mAYBE A FANTASY OR A MYSTERY? i SHOULD START WITH AN INTERESTING SETTING.

fIRST, i NEED A PROTAGONIST. pERHAPS A YOUNG CHARACTER WHO'S BRAVE OR CURIOUS. a GIRL NAMED lILA SOUNDS GOOD. sHE COULD BE FROM A SMALL VILLAGE, WHICH GIVES HER A SENSE OF ADVENTURE. tHE VILLAGE IS NEAR A MAGICAL FOREST, SO THAT SETS UP THE FANTASY ELEMENT.

nEXT, THE CONFLICT. mAYBE SOMETHING LIKE A MAGICAL CREATURE CAUSING TROUBLE. a DRAGON? bUT DRAGONS ARE USUALLY SCARY. mAYBE A MORE MYSTICAL CREATURE, LIKE A GUARDIAN OF THE FOREST. lET'S SAY A GUARDIAN NAMED tHARION. tHAT GIVES HER A NAME AND A PURPOSE.

tHE PROBLEM: tHE GUARDIAN IS TRAPPED IN AN ANCIENT TRAP, SO lILA NEEDS TO SOLVE IT. tHIS CREATES A QUEST. sHE HAS TO FIND THE KEY OR SOMETHING THAT CAN OPEN THE TRAP. mAYBE SHE NEEDS HELP FROM OTHERS, LIKE A FRIEND OR A MENTOR. bUT S

In [4]:
from langchain_core.runnables import RunnableGenerator


def streaming_parse(chunks: Iterable[AIMessageChunk]) -> Iterable[str]:
    for chunk in chunks:
        yield chunk.content.swapcase()


streaming_parse = RunnableGenerator(streaming_parse)

In [5]:
chain = model | streaming_parse
for chunk in chain.stream("tell me a story"):
    print(chunk, end="")

<THINK>
oKAY, THE USER WANTS A STORY. lET ME THINK ABOUT WHAT KIND OF STORY TO TELL. mAYBE SOMETHING WITH A TWIST OR A UNIQUE SETTING? a MAGICAL REALISM TYPE OF STORY COULD WORK. lET'S SEE... mAYBE A YOUNG GIRL AND HER GRANDMOTHER, LIVING IN A SMALL VILLAGE. tHE GRANDMOTHER HAS SOME MYSTERIOUS POWER, LIKE CONTROLLING TIME OR SOMETHING. bUT i NEED TO MAKE SURE IT'S NOT TOO CLICHÉ.

wAIT, THE USER MIGHT BE LOOKING FOR SOMETHING MORE RELATABLE. mAYBE A HEARTWARMING STORY WITH A LESSON? oR MAYBE A BIT OF SUSPENSE. lET ME BRAINSTORM... hOW ABOUT A GIRL WHO DISCOVERS A HIDDEN PLACE IN HER GRANDMOTHER'S ATTIC, AND THERE'S SOMETHING MAGICAL THERE. bUT i NEED TO MAKE IT NOT TOO SCARY. mAYBE A MIX OF MYSTERY AND KINDNESS.

aLTERNATIVELY, A STORY ABOUT FRIENDSHIP OR OVERCOMING ADVERSITY. mAYBE A CHARACTER FACING A CHALLENGE AND LEARNING FROM THEIR EXPERIENCES. lET ME THINK... a YOUNG GIRL NAMED lILA WHO FINDS AN OLD BOOK IN HER GRANDMOTHER'S ATTIC. tHE BOOK HAS STRANGE SYMBOLS THAT BRING HER VISI